## Prompt:

TASK DESCRIPTION:
  * This is an IMAGE-LEVEL BINARY CLASSIFICATION task implemented using an object detection model.
  * The goal is to determine whether an image contains an animal or not.

DATASET STRUCTURE:
  * DATASET_ROOT contains three subdirectories: train, test, and val.
  * Each directory contains two subdirectories:
  * images/ → contains image files (.jpg, .jpeg, .png)
  * labels/ → contains YOLO format .txt files

GROUND-TRUTH LOGIC: 
  * An image is considered an animal if a corresponding .txt file exists and is not empty in the labels/ folder.
  * A non-empty file is a file whose size is larger than 0, and the size of an empty image is 0.

MODEL REQUIREMENTS:
  * Use ONLY a pretrained Ultralytics YOLO detection model (e.g., yolov8n.pt).
  * Call our RESTful API for yolo inference.
  * Assume YOLO detects animals using class ID animal at index 0.

YOLO INFERENCE APIs:

  * Sample CURL Request:
    ```
    curl -sS -X POST '${BASEURL}/v1/yolo/infer' \
    -H 'Authorization: Bearer ${FLEXSERV_TOKEN}' \
    -H 'Content-Type: application/json' \
    -d '{"model":"${FLEXSERV_MODEL_ID}","task":"detect","source":{"type":"upload","media_type":"image","content_base64":"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL","filename":"NOR3__2019-07-19__11-40-00-1-_JPG.rf.b85ee30f99a803b09f8c5a7da7f9a508.jpg"},"params":{"conf":0.25,"iou":0.7,"imgsz":640,"max_det":300,"show_labels":true,"show_conf":true},"response":{"include":["predictions","timing"],"box_format":"xyxy","classification_topk":5,"return_original_shape":true}}' 
    ```

  * RESPONSE: 
    ```
    {
    "object": "yolo.inference",
    "task": "detect",
    "model": "/app/models/private/yolo--yolo26l/model.pt",
    "media_type": "image",
    "predictions": [
        {
        "frame_index": 0,
        "path": "image0.jpg",
        "original_shape": {
            "height": 640,
            "width": 640
        },
        "detections": [
            {
            "class_id": 61,
            "class_name": "toilet",
            "confidence": 0.480474591255188,
            "bbox": [
                0,
                29.76239013671875,
                637.2445068359375,
                629.2056274414062
            ],
            "bbox_format": "xyxy",
            "track_id": null
            }
        ]
        }
    ],
    "timing": {
        "inference_ms": 21.03
    },
    "annotated_media": null,
    "annotated_media_mime_type": null,
    "annotated_media_filename": null,
    "warnings": []
    }
    ```
For each image, one object in the 'predictions' array, any if anything detected, the 'detections' array will contain
a list of detected objects, and if nothing detected, there won't be 'detections' array. 
If any detected object is with class_id=0, an animal is detected. 


DETECTION LOGIC (IMPORTANT):

  * Run object detection on each image.
  * If the model produces AT LEAST ONE detection of an animal class with confidence >= 0.5 and IoU >= 0.7:
  *   → The image-level prediction is animal.

EVALUATION METRICS:

  * Iterate through the images in the test split.
  * Compare the image-level prediction with the ground truth (existence of label file).
  * Count: True Positives, True Negatives, False Positives, and False Negatives.

ACCURACY DEFINITION:

  * Overall accuracy = (True Positives + True Negatives) / Total Images

OUTPUT REQUIREMENTS:

  * Print for each image: filename, ground-truth status, and prediction.
  * At the end, print a summary report including total images, counts for each metric, and overall detection accuracy.

CODING REQUIREMENTS:

  * Store the main path in a global varaible DATASET_ROOT.
  * Set global variable for BASEURL and Bearer Auth Token (i.e. FLEXSERV_TOKEN).
  * Set global variable for BASE_YOLO_MODEL and FINE_TUNED_YOLO_MODEL, and also a MODEL_TO_USE for easy model switching.
  * Set global variable for confidence threashold and IoU threashold.
  * Make sure we disable SSL/TLS verification and also disable the related warning.
  * Make sure we pass image_name into the yolo inference request.
  * Make sure we pass the correct header for auth token and content-type in the final request.
  * Make sure we pass the request body correctly in the final request.
  * The `content_base64` field in the request should start with "data:image/jpeg;base64," and then appended with the base64 encoded image data.
  * Use pathlib or os for robust file path matching.
  * Read only .jpg files.
  * For inference of each image file, print the number of the image versus total number of images, the time spent for each inference request versus the total time spent for the entire inference step (in ms), the ground truth and detection result. 
  * Include clear comments explaining each step.
  * Output the accuracy in percentage format.
  * Don't use any mock or dummy functions. Make sure every line functions. 
  * It is okay to capture general Exception instead of every single type of Exceptions.

DEFENSIVE PROGRAMMING
  In case of any unexpected conditions, make sure the following: 
      1. Make sure we don't do SSL/TLS verification when sending request. 
      2. Make sure we avoid zero division

FACTS TO KNOW: 
  * BASEURL of FLEXSERV inference engine: https://vista.tacc.utexas.edu:60324
  * Bearer Auth token for FLEXSERV inference engine: 31b8148f20a4e8749dc232b48158a64b93ac7a988bd6aba5cc5de90c5654f984
  * FLEXSERV model ID format: FLEX:{PUB|PRI}:author/model[@revision], we only use private model pool, and omit the revision in model ID. 
  * DATASET_ROOT address: /home/jovyan/ai-tutorial-2026/datasets/AnimalEcology.v4i.yolov11
  * BASE_YOLO_MODEL for the request:  FLEX:PRI:yolo/yolo26n
  * FINE_TUNED_YOLO_MODEL for the request:  FLEX:PRI:yolo/yolo26n-fine-tuned

After the code, briefly explain how the program works in plain English.

## Sample code


In [4]:
import os
import requests
import json
import base64
from pathlib import Path
import time

# Global variables
DATASET_ROOT = Path('/home/jovyan/ai-tutorial-2026/datasets/AnimalEcology.v4i.yolov11')
IMAGE_DIR = DATASET_ROOT / 'test' / 'images'
LABEL_DIR = DATASET_ROOT / 'test' / 'labels'
BASEURL = 'https://vista.tacc.utexas.edu:60324'
FLEXSERV_TOKEN = '31b8148f20a4e8749dc232b48158a64b93ac7a988bd6aba5cc5de90c5654f984'
BASE_YOLO_MODEL = 'FLEX:PRI:yolo/yolo26n'
FINE_TUNED_YOLO_MODEL = 'FLEX:PRI:yolo/yolo26n-fine-tuned'
MODEL_TO_USE = BASE_YOLO_MODEL
CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.7

# Disable SSL warnings
requests.packages.urllib3.disable_warnings()

def get_image_files(directory):
    """Get all .jpg files in the given directory."""
    return list(directory.glob('*.jpg'))

def read_label_file(label_path):
    """Check if the label file exists and is not empty."""
    return label_path.exists() and label_path.stat().st_size > 0

def encode_image_to_base64(image_path):
    """Encode an image file to base64."""
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
    return f"data:image/jpeg;base64,{encoded_string}"

def perform_yolo_inference(image_path, model_id):
    """Perform YOLO inference on the given image."""
    image_name = image_path.name
    encoded_image = encode_image_to_base64(image_path)
    
    headers = {
        'Authorization': f'Bearer {FLEXSERV_TOKEN}',
        'Content-Type': 'application/json'
    }
    
    payload = {
        "model": model_id,
        "task": "detect",
        "source": {
            "type": "upload",
            "media_type": "image",
            "content_base64": encoded_image,
            "filename": image_name
        },
        "params": {
            "conf": CONF_THRESHOLD,
            "iou": IOU_THRESHOLD,
            "imgsz": 640,
            "max_det": 300,
            "show_labels": True,
            "show_conf": True
        },
        "response": {
            "include": ["predictions", "timing"],
            "box_format": "xyxy",
            "classification_topk": 5,
            "return_original_shape": True
        }
    }
    
    response = requests.post(f'{BASEURL}/v1/yolo/infer', headers=headers, json=payload, verify=False)
    return response.json()

def detect_animal_in_image(inference_result):
    """Detect if an animal is present in the image based on inference results."""
    predictions = inference_result.get('predictions', [])
    for prediction in predictions:
        detections = prediction.get('detections', [])
        for detection in detections:
            if detection['class_id'] == 0 and detection['confidence'] >= CONF_THRESHOLD:
                return True
    return False

def evaluate_model():
    """Evaluate the model on the test dataset."""
    image_files = get_image_files(IMAGE_DIR)
    total_images = len(image_files)
    true_positives = 0
    true_negatives = 0
    false_positives = 0
    false_negatives = 0
    total_time = 0
    
    for idx, image_path in enumerate(image_files, start=1):
        label_path = LABEL_DIR / (image_path.stem + '.txt')
        ground_truth = read_label_file(label_path)
        
        start_time = time.time()
        inference_result = perform_yolo_inference(image_path, MODEL_TO_USE)
        end_time = time.time()
        inference_time = (end_time - start_time) * 1000
        
        prediction = detect_animal_in_image(inference_result)
        
        print(f"Image {idx}/{total_images}: {image_path.name}, Ground Truth: {ground_truth}, Prediction: {prediction}, Inference Time: {inference_time:.2f} ms")
        
        total_time += inference_time
        
        if ground_truth and prediction:
            true_positives += 1
        elif not ground_truth and not prediction:
            true_negatives += 1
        elif ground_truth and not prediction:
            false_negatives += 1
        elif not ground_truth and prediction:
            false_positives += 1
    
    accuracy = (true_positives + true_negatives) / total_images * 100 if total_images > 0 else 0
    
    print("\nSummary Report:")
    print(f"Total Images: {total_images}")
    print(f"True Positives: {true_positives}")
    print(f"True Negatives: {true_negatives}")
    print(f"False Positives: {false_positives}")
    print(f"False Negatives: {false_negatives}")
    print(f"Total Inference Time: {total_time:.2f} ms")
    print(f"Overall Detection Accuracy: {accuracy:.2f}%")

if __name__ == "__main__":
    evaluate_model()


Summary Report:
Total Images: 0
True Positives: 0
True Negatives: 0
False Positives: 0
False Negatives: 0
Total Inference Time: 0.00 ms
Overall Detection Accuracy: 0.00%


## Paste your generated Code here 
